Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 03 · A BI model query

A BI model is a governed set of measures over datasets (BI_GUIDE.md). Through Flight you send an MQ query (fields, measures, order, limit) and get the model's answer as an Arrow table: the same figures as a BI report, with the model's row rules and masks for your person.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Notional, MTM and trades by desk

`trades-basic` is the simplest sample model: the trading pack's trades with four measures and a desk-to-book hierarchy. Fields are named `<dataset alias>.<column>`; measures by their names in the model.

In [ ]:
f = connect()
by_desk = f.model("trades-basic", {
    "fields": ["trade.DESK"],
    "measures": ["Notional", "MTM", "Trades"],
    "orderBy": [{"by": "Notional", "desc": True}],
    "limit": 50,
})
print("model:", meta(by_desk, "drishti.model"), "| masked:", meta(by_desk, "drishti.masked", "none"))
by_desk.to_pandas()

## Down the hierarchy: desk and book

Two fields group by both; the model decides how each measure adds up (MTM is a sum; a non-additive measure is never summed by the client).

In [ ]:
by_book = f.model("trades-basic", {
    "fields": ["trade.DESK", "trade.PORTFOLIO"],
    "measures": ["Trades", "MTM", "Average MTM per trade"],
    "orderBy": [{"by": "MTM", "desc": True}],
    "limit": 20,
})
by_book.to_pandas()

In [ ]:
f.close()